# Summer Curriculum - Week 1
## Autograd

This lesson is from the https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html website

When we talk about training neural networks, the most common way of going about this is back propogation. This essentially is the algorithm for computing the gradient of the loss function with respect to each parameter. Those gradients are then used by an optimizer to adjust the parameters (or the model weights) within the net.

The nice thing about PyTorch is that it has a built in differentiation engine called 'torch.autograd' which supports automatic computation of gradient for any computational graph.

I have gotten my feet wet with neural networks as a model only in a layman's terms, so this lesson is very new to me, but still makes sense so far based on the terminology used.

Moving forward we are going to be getting into how things are applied within a simple 1-layer nn where the input is 'x', the parameters (weights and bias) are 'w' and 'b', and there is some loss function. Everything will be defined in the below block.

In [ ]:
# Import necessary libraries
import torch

x = torch.ones(5) # This is the input tensor
y = torch.zeros(3) # This is the expected output
w = torch.randn(5, 3, requires_grad = True) # This is the weight tensor, initialized randomly and set to require gradients for backpropagation
b = torch.randn(3, requires_grad = True) # This is the bias tensor, initialized randomly and set to require gradients for backpropagation
z = torch.matmul(x, w) + b # These are the logits (pre-activation output), calculated as the matrix multiplication of x and w, plus the bias b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y) # This is the loss tensor, calculated using the binary cross-entropy loss function with logits

Like we said above 'w' and 'b' are the parameters of this network model that are to be optimized. To do this, it is required that we compute the gradient of the loss function with respect to these two parameters. To do this, it is necessary that we set requires_grad to 'True' like we did above for both 'w' and 'b'.

It is also worth saying that we can set the value of requires_grad to be 'True' when creating the tensor or later using 'x.requires_grad_(True)'.

In [2]:
print(f"Gradient function for z: {z.grad_fn}") # This prints the gradient function for z, which is the function that computes the gradient of z with respect to its inputs
print(f"Gradient function for loss: {loss.grad_fn}") # This prints the gradient function for loss, which is the function that computes the gradient of loss with respect to its inputs

Gradient function for z: <AddBackward0 object at 0x10ba1b850>
Gradient function for loss: <BinaryCrossEntropyWithLogitsBackward0 object at 0x10ba1b130>


When we talk about optimizing parameters in a nn, we are talking about computing the derivatives of the loss function with respect to the specific parameters, under some fixed values of 'x' and 'y'. To do this, we use the loss.backward() function call and then retrieve the values from w.grad and b.grad

In [3]:
loss.backward() # This computes the gradients of the loss with respect to all tensors that have requires_grad=True, which in this case are w and b
print(w.grad) # This prints the gradient of the loss with respect to the weight tensor w
print(b.grad) # This prints the gradient of the loss with respect to the bias tensor b

tensor([[0.2974, 0.1428, 0.2827],
        [0.2974, 0.1428, 0.2827],
        [0.2974, 0.1428, 0.2827],
        [0.2974, 0.1428, 0.2827],
        [0.2974, 0.1428, 0.2827]])
tensor([0.2974, 0.1428, 0.2827])


We can only get grad properties from leaf nodes that were initialized with the requires_grad value set to 'True'.

We can only perform gradient calculations using backward once on a given graph for performance reasons. If we need to do several backward calls on the same graph, it is necessary to pass retain_graph = True to the backward call.

We use the requires_grad = True method when we want to keep track of the computational history and to support gradient computation. There are some cases though where this is not necessary, such as when we have an already trained model and we are just looking to apply some input data. This is a situation where we only care about the forward computation of the nn, so we can disable the tracking using torch.no_grad()

In [5]:
z = torch.matmul(x, w) + b # This is the logits (pre-activation output), calculated as the matrix multiplication of x and w, plus the bias b
print(z.requires_grad) # This prints whether the tensor z requires gradients, which is True in this case because it is a result of operations involving tensors that require gradients (w and b)

with torch.no_grad():
    z = torch.matmul(x, w) + b # This is the logits (pre-activation output), calculated as the matrix multiplication of x and w, plus the bias b
    print(z.requires_grad) # This prints whether the tensor z requires gradients, which is False in this case because it is computed within a no_grad context, meaning that it will not track operations for gradient computation

True
False


We can also do this using the detach() method on the tensor.

In [6]:
z = torch.matmul(x, w) + b # Recompute the logits after the gradients have been calculated
z_det = z.detach()
print(z_det.requires_grad) # This prints whether the detached tensor z_det requires gradients, which is False because it has been detached from the computation graph, meaning that it will not track operations for gradient computation

False


Some reasons that we may want to disablegradient tracking are:

- To mark some parameters in a nn as frozen parameters
- To speed up computations when only doing a forward pass, because adding the tracking is less efficient

Autograd keeps a record of the data (tensors) and the executed operations along with the new resultant tensors in something called a 'directed acyclic graph' or DAG, consisting of function objects. In the DAG, the leaves are the input tensors and the roots are the output tensors. When we trace this graph from roots to leaves we can automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

- Runs the requested operation to compute the resultant tensor
- Maintains the operation's gradient function in the DAG

The backward pass kicks off when .backward() iscalled on the DAG root. Autograd then:

- Computes the gradients from each
- Accumulates them in the respective tensor's .grad attribute
- Using the chain rule, propogates all the way to the leaf tensors

Jacobian Products???

In [7]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


### What is actually happening here

When the output of our network is a scalar (like a single loss value), `.backward()` is straightforward — the "gradient with respect to inputs" is a single tensor of partial derivatives, one per input element. That is what we have been computing for `w` and `b` up to this point.

But when the output is **not** a scalar — like the 5×4 matrix `out` here — the "gradient" is no longer a single set of partial derivatives. It is a whole matrix of them, called the **Jacobian**:

$$J_{ij} = \frac{\partial \text{out}_i}{\partial \text{inp}_j}$$

For an output with m elements and an input with n elements, the Jacobian is an m×n matrix where each entry tells us how output element i changes when input element j is nudged a tiny bit. Our scalar-output gradients from earlier are just a special case of this when m = 1 — the Jacobian collapses to a single row.

### Why autograd does not just compute the full Jacobian

Building the full Jacobian for a real network would be extremely expensive (output size × input size entries to store). PyTorch is instead built around a cheaper primitive called a **Jacobian-vector product**: Jᵀ · v, which gives back a single tensor the same shape as the input.

The key sentence to internalize:

> `out.backward(v)` computes the gradient of the scalar `sum(v * out)` with respect to `inp`.

So passing `v = torch.ones_like(out)` (a tensor of all ones, like we did above) is equivalent to computing the gradient of `sum(out)` with respect to `inp`. That gives us one weighted "total gradient" without privileging any particular output element.

### Walking through the math by hand

In the cell above:
- `inp = torch.eye(4, 5)` is a 4×5 matrix with 1s on the diagonal and 0s elsewhere
- `out = (inp + 1).pow(2).t()` adds 1 to every element, squares each, then transposes
- `out.backward(torch.ones_like(out))` is asking for the gradient of `sum((inp+1)²)` with respect to `inp` (the transpose does not change the sum, so we can ignore it for the derivative)

By the power rule from calculus, the partial derivative of `(inp[i,j] + 1)²` with respect to `inp[i,j]` is `2 · (inp[i,j] + 1)`:
- For diagonal elements where `inp[i,j] = 1`: `2 · 2 = 4`
- For off-diagonal elements where `inp[i,j] = 0`: `2 · 1 = 2`

That matches the first-call output exactly. No magic — it is just the chain rule applied to a small example.

### Gradients accumulate; they do not overwrite

The second call doubles every entry of `.grad` even though we asked for the same computation. That is because **PyTorch accumulates gradients** — every `.backward()` call adds to whatever is already in `.grad`. It does not replace.

That is why `inp.grad.zero_()` was called between the second and third backward calls — it wipes `.grad` back to zero so the third call returns to the first-call values.

This is intentional. It enables:
- **Gradient accumulation across micro-batches** — split a too-big batch into chunks, call `.backward()` on each, then run one optimizer step. Equivalent to a single big-batch backward with smaller memory footprint per step.
- **Shared parameters** — if a weight is used multiple times in one forward pass (RNNs, weight tying, etc.), the gradient contributions from each use accumulate into one `.grad`, which is what the chain rule mathematically requires.

### Why this matters for normal training

The accumulation default is the reason every PyTorch training loop has this pattern:

```python
optimizer.zero_grad()    # wipe .grad before backward
loss.backward()          # compute new gradients (added to now-empty .grad)
optimizer.step()         # use them to update parameters
```

Without `zero_grad()`, gradients from the previous batch would still be sitting in `.grad`, and the next backward call would add to them — polluting the training signal with stale information. This is one of the classic beginner bugs in PyTorch, and we now know exactly what causes it.